# 🔬 Physics-Informed Residual Learning (v4)
**Concept:** Physics baseline (linear extrapolation) + XGBoost learns biological residual.  
`Linear_Forecast = CGM + Vel_15min × 2`  
`Target_Residual = Target_CGM − Linear_Forecast`  
`Final_Pred = Linear_Forecast + Predicted_Residual`

---

In [ ]:
!pip install polars xgboost plotly pyarrow scikit-learn -q
import polars as pl, numpy as np, xgboost as xgb
import plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error
from datetime import datetime, timedelta
import os, gc, time, json

from google.colab import drive
drive.mount('/content/drive')
print(f"XGBoost: {xgb.__version__}")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
BASE   = "/content/drive/MyDrive/DiabetesKDS/data"
INPUT  = f"{BASE}/processed/train_elite_physio_engineered.parquet"
MODEL  = f"{BASE}/models/xgboost_physics_residual_v4.json"
os.makedirs(os.path.dirname(MODEL), exist_ok=True)

FEATURES = ["CGM","IOB","COB","Vel_5min","Vel_15min","Acc_15min",
            "Roll_Mean_60","Roll_Std_60","Hour_Sin","Hour_Cos",
            "Insulin_Sensitivity_Risk"]
print(f"📁 {INPUT} — exists: {os.path.exists(INPUT)}")

---\n## STEP 1 — Physics Baseline + Residual Target

In [ ]:
df = pl.read_parquet(INPUT)
FEATURES = [c for c in FEATURES if c in df.columns]
print(f"Loaded: {df.shape[0]:,} rows")

# Physics baseline: linear extrapolation using 15-min velocity
# Vel_15min covers 3 steps (15 min). Forecast 6 steps (30 min) = 2× velocity.
df = df.with_columns([
    (pl.col("CGM") + pl.col("Vel_15min") * 2).alias("Linear_Forecast"),
])
df = df.with_columns(
    (pl.col("Target_CGM") - pl.col("Linear_Forecast")).alias("Target_Residual")
)

# Cleanup
df = df.sort("datetime").drop_nulls(
    subset=FEATURES + ["Target_Residual", "Target_CGM", "Linear_Forecast"]
)
print(f"After cleanup: {len(df):,} rows")

# Residual statistics
r = df["Target_Residual"]
print(f"\nTarget_Residual stats:")
print(f"  Mean: {r.mean():.4f}  Std: {r.std():.2f}  Min: {r.min():.1f}  Max: {r.max():.1f}")

# How good is the physics baseline alone?
lf = df["Linear_Forecast"].to_numpy()
tc = df["Target_CGM"].to_numpy()
baseline_rmse = np.sqrt(np.mean((lf - tc)**2))
baseline_mae  = np.mean(np.abs(lf - tc))
print(f"\n📐 Physics Baseline alone:")
print(f"   RMSE: {baseline_rmse:.4f}  MAE: {baseline_mae:.4f}")

In [ ]:
# Residual distribution
fig = make_subplots(rows=1, cols=2, subplot_titles=("Target Residual", "Linear Forecast Error"))
fig.add_trace(go.Histogram(x=df["Target_Residual"].clip(-60,60).to_numpy(),
    nbinsx=200, marker_color="#9B59B6", name="Residual"), row=1, col=1)
fig.add_trace(go.Histogram(x=np.clip(lf-tc, -60, 60),
    nbinsx=200, marker_color="#E74C3C", name="Baseline Err"), row=1, col=2)
fig.update_layout(template="plotly_dark", height=350,
    title="📊 What XGBoost needs to learn (residual) vs Physics baseline error")
fig.show()

In [ ]:
# ── Time-Series Split ────────────────────────────────────────────────
split = int(len(df) * 0.80)
tr, te = df.head(split), df.tail(len(df) - split)

X_train = tr.select(FEATURES).to_numpy().astype(np.float32)
y_train = tr["Target_Residual"].to_numpy().astype(np.float32)
y_train_abs = tr["Target_CGM"].to_numpy().astype(np.float32)
lf_train = tr["Linear_Forecast"].to_numpy().astype(np.float32)

X_test = te.select(FEATURES).to_numpy().astype(np.float32)
y_test_res = te["Target_Residual"].to_numpy().astype(np.float32)
y_test_abs = te["Target_CGM"].to_numpy().astype(np.float32)
lf_test = te["Linear_Forecast"].to_numpy().astype(np.float32)
cgm_test = te["CGM"].to_numpy().astype(np.float32)
test_ids = te["id"].to_list()
test_dt = te["datetime"].to_list()

del df, tr; gc.collect()
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

---\n## STEP 2 — Clinical Weights + Training

In [ ]:
# Weights based on ABSOLUTE target (not residual)
def cw(y):
    w = np.ones_like(y, dtype=np.float32)
    w[y < 70] = 10.0; w[y > 250] = 5.0; w[(y>=180)&(y<=250)] = 2.0
    return w

w_tr = cw(y_train_abs); w_te = cw(y_test_abs)
for l,c in [("Hypo 10x",w_tr==10),("Hyper 5x",w_tr==5),("Mild 2x",w_tr==2),("Normal",w_tr==1)]:
    print(f"  {l:14s}: {c.sum():>10,} ({c.sum()/len(w_tr)*100:.2f}%)")

In [ ]:
model = xgb.XGBRegressor(
    objective="reg:squarederror", tree_method="hist", device="cuda",
    n_estimators=3000, early_stopping_rounds=50,
    eval_metric="rmse", random_state=42, verbosity=1,
    learning_rate=0.02, max_depth=10,
    subsample=0.8, colsample_bytree=0.9,
    reg_alpha=1.0, reg_lambda=5.0, min_child_weight=10,
)

print("🚀 Training residual model...")
t0 = time.time()
model.fit(
    X_train, y_train, sample_weight=w_tr,
    eval_set=[(X_train, y_train), (X_test, y_test_res)],
    sample_weight_eval_set=[w_tr, w_te], verbose=200,
)
t_train = time.time() - t0
print(f"\n✅ Done in {t_train:.1f}s | Best iter: {model.best_iteration}")

In [ ]:
res = model.evals_result()
fig = go.Figure()
fig.add_trace(go.Scatter(y=res["validation_0"]["rmse"], name="Train", line=dict(color="#4ECDC4")))
fig.add_trace(go.Scatter(y=res["validation_1"]["rmse"], name="Test", line=dict(color="#FF6B6B")))
fig.add_vline(x=model.best_iteration, line_dash="dash", line_color="white")
fig.update_layout(title=f"📉 Residual Model Curves (best={model.best_iteration})",
    template="plotly_dark", height=370, xaxis_title="Round", yaxis_title="RMSE")
fig.show()

---\n## STEP 3 — Reconstruct & Evaluate

In [ ]:
# RECONSTRUCT: Final = Physics_Baseline + Learned_Residual
pred_res = model.predict(X_test)
pred_final = np.clip(lf_test + pred_res, 20, 500)

# Also compute: what if we just used physics baseline?
physics_only = np.clip(lf_test, 20, 500)

# Metrics
rmse_r     = np.sqrt(mean_squared_error(y_test_res, pred_res))
rmse_final = np.sqrt(mean_squared_error(y_test_abs, pred_final))
mae_final  = mean_absolute_error(y_test_abs, pred_final)
rmse_phys  = np.sqrt(mean_squared_error(y_test_abs, physics_only))
m = y_test_abs > 0
mape = np.mean(np.abs((y_test_abs[m] - pred_final[m]) / y_test_abs[m])) * 100
errs = np.abs(y_test_abs - pred_final)
w10, w15, w20 = [np.mean(errs <= t)*100 for t in [10,15,20]]

print("╔═══════════════════════════════════════════════════════════╗")
print("║         📊  PHYSICS-INFORMED v4 METRICS                 ║")
print("╠═══════════════════════════════════════════════════════════╣")
print(f"║  Physics baseline RMSE   : {rmse_phys:>8.4f} mg/dL            ║")
print(f"║  Residual model RMSE     : {rmse_r:>8.4f} mg/dL (on resid)  ║")
print(f"║  ─────────────────────────────────────────────────────   ║")
print(f"║  FINAL RMSE (recon)      : {rmse_final:>8.4f} mg/dL            ║")
print(f"║  FINAL MAE               : {mae_final:>8.4f} mg/dL            ║")
print(f"║  MAPE                    : {mape:>8.4f}%                    ║")
print(f"║  Within ±10              : {w10:>8.2f}%                      ║")
print(f"║  Within ±15              : {w15:>8.2f}%                      ║")
print(f"║  Within ±20              : {w20:>8.2f}%                      ║")
print(f"║                                                           ║")
print(f"║  Improvement over physics: {rmse_phys - rmse_final:>+8.4f} mg/dL RMSE      ║")
print("╚═══════════════════════════════════════════════════════════╝")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CLARKE ERROR GRID
# ══════════════════════════════════════════════════════════════════════
def clarke_zones(ref, pred):
    ref, pred = np.asarray(ref,float), np.asarray(pred,float)
    z = np.full(len(ref), 'D', dtype='U1')
    for i in range(len(ref)):
        r,p = ref[i], pred[i]
        if (r<=70 and p<=70) or abs(p-r)<=0.2*r: z[i]='A'
        elif (r<=70 and p>=180) or (r>=180 and p<=70): z[i]='E'
        elif 70<=r<=180 and (p<70 or p>180): z[i]='C'
        elif (r<70 and 70<p<=180) or (r>180 and 70<=p<180): z[i]='D'
        else: z[i]='B'
    return z

# Compute for final prediction AND physics-only baseline
z_final = clarke_zones(y_test_abs, pred_final)
z_phys  = clarke_zones(y_test_abs, physics_only)

def zone_report(z, label):
    uq, ct = np.unique(z, return_counts=True)
    zd = dict(zip(uq, ct)); t = len(z)
    ab = (zd.get('A',0)+zd.get('B',0))/t*100
    print(f"  {label}: A+B={ab:.2f}%  D={zd.get('D',0)/t*100:.2f}%  E={zd.get('E',0)/t*100:.3f}%")
    return zd, t, ab

print("Clarke Error Grid Comparison:")
zd_f, total, zone_ab = zone_report(z_final, "Physics+Residual (v4)")
zd_p, _, zone_ab_p   = zone_report(z_phys,  "Physics Only (baseline)")

print(f"\n╔═══════════════════════════════════════════════════╗")
print(f"║       🩸  CLARKE v4 (Physics + Residual)          ║")
print(f"╠═══════════════════════════════════════════════════╣")
for zone in ['A','B','C','D','E']:
    n=zd_f.get(zone,0); pct=n/total*100
    bar='█'*int(pct/2)
    print(f"║  Zone {zone}: {n:>10,} ({pct:>6.2f}%) {bar:20s}  ║")
print(f"║                                                   ║")
s = "✅ PASS" if zone_ab > 95 else "⚠️  BELOW"
print(f"║  A+B = {zone_ab:.2f}%  (target >95%)  {s:12s}    ║")
print(f"║  v1=92.91%  →  Δ = {zone_ab-92.91:+.2f}%                    ║")
print(f"╚═══════════════════════════════════════════════════╝")

In [ ]:
# Clarke Grid Plot
sn = min(100000, total)
ix = np.random.RandomState(42).choice(total, sn, replace=False)
zs = clarke_zones(y_test_abs[ix], pred_final[ix])
cl = {'A':'#2ECC71','B':'#F39C12','C':'#E74C3C','D':'#9B59B6','E':'#C0392B'}

fig = go.Figure()
fig.add_trace(go.Scattergl(x=y_test_abs[ix], y=pred_final[ix], mode='markers',
    marker=dict(size=2, color=[cl.get(z,'gray') for z in zs], opacity=0.4), hoverinfo='skip'))
fig.add_trace(go.Scatter(x=[0,400],y=[0,400], mode='lines',
    line=dict(color='white',width=1,dash='dash'), name='Perfect'))
xl = np.linspace(1,400,300)
fig.add_trace(go.Scatter(x=xl,y=xl*1.2, mode='lines',
    line=dict(color='rgba(46,204,113,0.4)'), showlegend=False))
fig.add_trace(go.Scatter(x=xl,y=xl*0.8, mode='lines',
    line=dict(color='rgba(46,204,113,0.4)'), showlegend=False))
for v in [70,180]:
    fig.add_hline(y=v, line_dash='dot', line_color='rgba(255,255,255,0.2)')
    fig.add_vline(x=v, line_dash='dot', line_color='rgba(255,255,255,0.2)')
fig.add_annotation(x=340,y=50, text=f"A+B: {zone_ab:.1f}%",
    font=dict(size=16,color='white'), showarrow=False)
fig.update_layout(title="🩸 Clarke Grid — Physics-Informed Residual v4",
    xaxis_title="Reference", yaxis_title="Predicted",
    template="plotly_dark", height=600, width=600,
    xaxis=dict(range=[0,400]), yaxis=dict(range=[0,400]))
fig.show()

---\n## STEP 4 — Feature Importance + Bias Check

In [ ]:
imp = model.get_booster().get_score(importance_type="gain")
fi = {FEATURES[int(k.replace('f',''))]: v for k,v in imp.items() if int(k.replace('f',''))<len(FEATURES)}
fi = dict(sorted(fi.items(), key=lambda x:x[1], reverse=True))
tg = sum(fi.values())

print("╔══════════════════════════════════════════════════════╗")
print("║    🏆  FEATURE IMPORTANCE (Residual Model)         ║")
print("╠══════════════════════════════════════════════════════╣")
for n,v in fi.items():
    p = v/tg*100; bar='█'*int(p)
    print(f"║  {n:28s} {p:>6.2f}%  {bar:18s}  ║")
print("╚══════════════════════════════════════════════════════╝")

cgm_p = fi.get('CGM',0)/tg*100
print(f"\n🔍 CGM share: {cgm_p:.1f}%  (v1=~80%, v3=delta)")
print(f"   {'✅ Bias reduced — model uses biology!' if cgm_p < 50 else '⚠️ Still CGM-heavy'}")

fig = px.bar(x=list(fi.values()), y=list(fi.keys()), orientation='h',
    title='🏆 Feature Importance — Physics-Informed Residual v4',
    color=list(fi.values()), color_continuous_scale='Viridis')
fig.update_layout(template='plotly_dark', height=400, yaxis=dict(autorange='reversed'))
fig.show()

In [ ]:
# ── 24h Prediction Trace ─────────────────────────────────────────────
ta = np.array(test_ids)
uid,uc = np.unique(ta, return_counts=True)
bp = uid[np.argmax(uc)]; pm = ta==bp
pdt = np.array(test_dt)[pm]
mid = len(pdt)//2
wm = (pdt>=pdt[mid]) & (pdt<=pdt[mid]+timedelta(hours=24))

fig = go.Figure()
fig.add_trace(go.Scatter(x=pdt[wm], y=y_test_abs[pm][wm], name='Actual',
    line=dict(color='#00D4AA', width=2.5)))
fig.add_trace(go.Scatter(x=pdt[wm], y=pred_final[pm][wm], name='Physics+Residual',
    line=dict(color='#FF6B6B', width=2, dash='dot')))
fig.add_trace(go.Scatter(x=pdt[wm], y=physics_only[pm][wm], name='Physics Only',
    line=dict(color='#9B59B6', width=1.5, dash='dash')))
fig.add_hline(y=70, line_dash='dash', line_color='red')
fig.add_hline(y=180, line_dash='dash', line_color='orange')
fig.update_layout(title=f"📉 24h Trace — Patient '{bp}'",
    template='plotly_dark', height=450, xaxis_title='Time', yaxis_title='mg/dL')
fig.show()

---\n## STEP 5 — Save & Evolution Report

In [ ]:
model.save_model(MODEL)
msz = os.path.getsize(MODEL)/(1024**2)
meta = {
    "model": "xgboost_physics_residual_v4",
    "strategy": "physics_informed_residual_learning",
    "physics_baseline": "CGM + Vel_15min * 2",
    "xgboost_target": "Target_Residual = Target_CGM - Linear_Forecast",
    "reconstruction": "Final = Linear_Forecast + Predicted_Residual",
    "created": datetime.now().isoformat(),
    "features": FEATURES, "horizon_min": 30,
    "best_iteration": int(model.best_iteration),
    "metrics": {
        "physics_rmse": round(float(rmse_phys),4),
        "residual_rmse": round(float(rmse_r),4),
        "final_rmse": round(float(rmse_final),4),
        "final_mae": round(float(mae_final),4),
        "mape": round(float(mape),4),
        "clarke_ab_pct": round(float(zone_ab),2),
        "zone_d_pct": round(zd_f.get('D',0)/total*100, 2),
    },
    "cgm_importance_pct": round(float(cgm_p),2),
}
with open(MODEL.replace('.json','_metadata.json'),'w') as f:
    json.dump(meta, f, indent=2)
print(f"💾 Model: {MODEL} ({msz:.1f} MB)")

In [ ]:
print("")
print("╔══════════════════════════════════════════════════════════════════╗")
print("║            🏁  MODEL EVOLUTION REPORT                         ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║            v1(abs)  v3(delta)  v4(phys+res)                   ║")
print("║  ────────────────────────────────────────────────────────────  ║")
print(f"║  Clarke A+B:  92.91%      —       {zone_ab:>6.2f}%                    ║")
print(f"║  Zone D:       4.80%      —       {zd_f.get('D',0)/total*100:>6.2f}%                    ║")
print(f"║  CGM share:    ~80%       —       {cgm_p:>6.1f}%                    ║")
print(f"║  RMSE:          —         —       {rmse_final:>8.4f}                  ║")
print("║                                                                ║")
print("║  Strategy: Linear_Forecast + XGBoost(biological residual)     ║")
print(f"║  Physics alone RMSE: {rmse_phys:.4f}  →  +Residual: {rmse_final:.4f}         ║")
print(f"║  Improvement:  {rmse_phys-rmse_final:+.4f} mg/dL RMSE                        ║")
print("║                                                                ║")
v1 = "✅ TARGET MET" if zone_ab > 95 else "⚠️  Continue iterating"
v2 = "✅ BIAS BROKEN" if cgm_p < 50 else "⚠️  CGM still heavy"
print(f"║  CLARKE:  {v1:53s} ║")
print(f"║  BIAS:    {v2:53s} ║")
print("╚══════════════════════════════════════════════════════════════════╝")
gc.collect()
print("\n🧹 Done.")